https://adk.dev/agents/custom-agents/

In [8]:
import os
from dotenv import load_dotenv
load_dotenv()

os.environ["GEMINI_API_KEY"] = os.getenv("GEMINI_API_KEY1")

Para construir nuestra propia lógica, necesitamos heredar de `BaseAgent` y definir el método `_run_async_impl`

# Construcción de la arquitectura

In [ ]:
from google.adk.agents import BaseAgent, LlmAgent
from google.adk.agents.invocation_context import InvocationContext
from google.adk.events import Event
from google.genai import types

from typing import override, AsyncGenerator

import logging
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)


class AgenteMatematico(BaseAgent):

    # Le decimos a Pydantic que variables acepta nuestra arquitectura
    sumas: LlmAgent
    restas: LlmAgent

    # Le decimos a Pydantic que nuestras variables pueden ser de cualquier tipo (LlmAgent en este caso)
    model_config = {"arbitrary_types_allowed": True}

    def __init__(self, name: str, sumas: LlmAgent, restas: LlmAgent):

        sub_agents_list = [sumas, restas]
        super().__init__(name=name, sumas=sumas, restas=restas, sub_agents=sub_agents_list)
    
    @override
    async def _run_async_impl(self, ctx: InvocationContext) -> AsyncGenerator[Event, None]:
        
        logger.info(f"[{self.name}] Empezando el flujo")

        user_input = ctx.user_content.parts[0].text
        logger.info(f"USER INPUT: {user_input}")
        if "+" in user_input:
            chosen_agent = self.sumas
            logger.info(f"[{chosen_agent.name}] Haciendo la suma")
        elif "-" in user_input:
            chosen_agent = self.restas
            logger.info(f"[{chosen_agent.name}] Haciendo la resta")
        else:
            yield Event(author=self.name, content=types.Content(parts=[types.Part.from_text(text="Solo sé sumar o restar")]))
        
        async for event in chosen_agent.run_async(ctx):
            logger.info(f"[Events] {event.model_dump_json(indent=2, exclude_none=True)}")
            yield event

        logger.info("Agent Run terminado")

# Creamos los Agentes

In [10]:
APP_NAME = "pruebas_custom_agent"
USER_ID = "12345"
SESSION_ID = "123344"
GEMINI_2_FLASH = "gemini-2.5-flash-lite"

def sumar(a: int, b: int):
    """Herramienta para hacer la suma de dos números"""
    return a + b
sumas = LlmAgent(
    name = "AgenteSumas",
    model = GEMINI_2_FLASH,
    instruction = """Eres una persona experta en hacer sumas de números naturales. Usa la herramienta para realizar esta operación y, una vez tengas el resultado, formula una respuesta de texto para el usuario""",
    output_key = "sumas_resp",
    tools=[sumar]
)

def restar(a: int, b: int):
    """Herramienta para hacer la resta de dos números"""
    return a - b
restas = LlmAgent(
    name = "AgenteRestas",
    model = GEMINI_2_FLASH,
    instruction = """Eres una persona experta en hacer restas de números naturales. Usa la herramienta para realizar esta operación y, una vez tengas el resultado, formula una respuesta de texto para el usuario""",
    output_key = "restas_resp",
    tools=[restar]
)


agente_matematico = AgenteMatematico(
    name = "AgenteMatematico",
    sumas = sumas,
    restas = restas
)

# Establecemos Sesión y Runner

In [11]:
from google.adk.sessions import InMemorySessionService
from google.adk.runners import Runner

async def setup_session_and_runner():
    session_service = InMemorySessionService()
    session = await session_service.create_session(app_name=APP_NAME, user_id=USER_ID, session_id=SESSION_ID)
    runner = Runner(
        agent=agente_matematico,
        app_name=APP_NAME,
        session_service=session_service
    )
    return session_service, runner

async def call_agent_async(user_input: str):

    session_service, runner = await setup_session_and_runner()

    current_session = session_service.sessions[APP_NAME][USER_ID][SESSION_ID]

    content = types.Content(role="user", parts=[types.Part(text=user_input)])

    events = runner.run_async(user_id=USER_ID, session_id=SESSION_ID, new_message=content)
    async for event in events:
        if event.is_final_response() and event.content and event.content.parts:
            final_response = event.content.parts[0].text

    final_session = await session_service.get_session(app_name=APP_NAME, user_id=USER_ID, session_id=SESSION_ID)
    print("ESTADO FINAL DE LA SESIÓN:\n", final_session.state)

In [12]:
await call_agent_async("Cuánto es 2 + 2?")

INFO:__main__:[AgenteMatematico] Empezando el flujo
INFO:__main__:USER INPUT: Cuánto es 2 + 2?
INFO:__main__:[AgenteSumas] Haciendo la suma
INFO:google_adk.google.adk.models.google_llm:Sending out request, model: gemini-2.5-flash-lite, backend: GoogleLLMVariant.GEMINI_API, stream: False
INFO:google_adk.google.adk.models.google_llm:Response received from the model.
INFO:__main__:[Events] {
  "model_version": "gemini-2.5-flash-lite",
  "content": {
    "parts": [
      {
        "function_call": {
          "id": "adk-527ff75f-da5e-4638-bcb8-e60b2393ec71",
          "args": {
            "a": 2,
            "b": 2
          },
          "name": "sumar"
        }
      }
    ],
    "role": "model"
  },
  "finish_reason": "STOP",
  "usage_metadata": {
    "candidates_token_count": 19,
    "prompt_token_count": 113,
    "prompt_tokens_details": [
      {
        "modality": "TEXT",
        "token_count": 113
      }
    ],
    "total_token_count": 132
  },
  "invocation_id": "e-274db014-806

ESTADO FINAL DE LA SESIÓN:
 {'sumas_resp': '2 + 2 es igual a 4.'}
